In [ ]:
!pip install edsl

# Augmentation

In [ ]:
from edsl import login
login()

In [ ]:
# =========================================================
# Varying Assistant Scaffolds (Fixed Worker)
# =========================================================
from edsl import Agent, Model, Scenario, ScenarioList, Survey, QuestionFreeText
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------
# 1. Experiment Configuration
# ---------------------------------------------------------
BASE_WORKER_MODEL = "gpt-3.5-turbo"   # fixed base model (worker)
EVALUATOR_MODEL = "gpt-5"             # for later grading
N_REPLICATES = 1                     # replicates per condition per assistant

ASSISTANT_MODELS = {
    "gpt-4.1": "GPT-4.1",
    "deepseek-ai/DeepSeek-V3.1": "DeepSeek-V3.1",
    "o4-mini-2025-04-16": "GPT-O4-Mini",
    "o3-mini-2025-01-31": "GPT-O3-Mini",
    "openai/gpt-oss-120b": "GPT-OSS-120B",
    "anthropic.claude-3-5-sonnet-20240620-v1:0": "Claude-3.5-Sonnet",
    "gpt-5.4": "GPT-5.4",                          # verify via Model.available()
    "claude-opus-4-6": "Claude-Opus-4.6",    # verify via Model.available()
    "gemini-3.1-pro-preview": "Gemini-3.1-Pro",     # verify via Model.available()
}

# ---------------------------------------------------------
# 2–7: Everything below is UNCHANGED from your code
# ---------------------------------------------------------
TASK_NAME = "travel_agent"
TASK_PROMPT = """
A customer tells you they want to take a 5-day trip to Tokyo next month with a $2,500 budget.

In one comprehensive response, do ALL of the following:

(1) Ask any missing clarifying questions needed to finalize plans.

(2) Compute estimated costs for flights, hotels, local transportation, and activities.

(3) Provide 2–3 hotel options with nightly rates.

(4) Suggest a transportation plan (airline choices, rail passes, transit tips).

(5) Provide brochures-style info: local customs, key points of interest, cultural etiquette, necessary regulations, passport/visa notes.

(6) Create a detailed 5-day itinerary with morning/afternoon/evening activities.

(7) Offer at least one promotional package or bundled tour option.

(8) Present a clear cost breakdown and confirm whether the trip fits the budget.

Respond exactly as a professional travel agent would. Be thorough, professional, and customer-focused. Provide realistic and practical recommendations that fit within the customer's budget.
"""

ASSISTANT_PROMPT_TEMPLATE = """
You are a planning coach helping another model prepare to complete a task.

Your role is to provide PROCESS guidance only, not task content.

Write a short 200–250 word guidance message that gives a three-phase workflow:

1. Requirements Check
- identify the deliverable, audience, tone, and constraints
- note what a strong answer must include

2. Plan & Structure
- suggest a general response structure or sequence of sections
- suggest how to balance completeness, clarity, and tone

3. Draft -> Self-Review -> Finalize
- suggest a brief checklist for revising the answer before submission

Important rules:
- Do NOT solve the task
- Do NOT provide topic-specific content, examples, or recommendations
- Do NOT write sentences that could be copied into the final answer
- Do NOT restate the task as a completed outline with filled-in content
- Keep the guidance generic and process-focused

Format in Markdown under the heading:
**Assistant Guidance: Three-Phase Workflow**
"""

def generate_scaffold(task_prompt, assistant_model):
    assistant_agent = Agent(instruction=ASSISTANT_PROMPT_TEMPLATE)
    q = QuestionFreeText("scaffold", "{{ scenario.task_prompt }}")
    survey = Survey([q])
    scenario = Scenario({"task_prompt": task_prompt})
    print(f"\n🧩 Generating scaffold from {assistant_model}...")
    results = survey.by(scenario).by(assistant_agent).by(Model(assistant_model, max_tokens=4096)).run()
    scaffold_text = results.select("scaffold").to_list()[0]
    print(f"✅ Scaffold generated from {assistant_model[:40]}...")
    return scaffold_text

def run_condition(base_worker_model, condition, task_prompt, scaffold_text=None):
    full_prompt = f"{scaffold_text}\n\n{task_prompt}" if scaffold_text else task_prompt
    scenarios = ScenarioList([
        Scenario({
            "task_prompt": full_prompt,
            "condition": condition,
            "replicate_id": i
        })
        for i in range(N_REPLICATES)
    ])
    q = QuestionFreeText("output", "{{ scenario.task_prompt }}")
    survey = Survey([q])
    worker = Agent(
        instruction=(
            "If the task prompt includes an 'Assistant Guidance: Three-Phase Workflow' section at the top, "
            "read it carefully first and follow its planning and self-review steps before producing your response. "
            "Then complete the task itself clearly and professionally. "
            "Return only the final deliverable (the completed plan) — do not restate the guidance."
        )
    )
    print(f"🚀 Running {base_worker_model} [{condition}]...")
    results = (
        survey.by(scenarios)
        .by(worker)
        .by(Model(base_worker_model))
        .run(
            n=1, progress_bar=True, verbose=True,
            remote_inference_description=f"{TASK_NAME}-{base_worker_model}-{condition}",
            remote_inference_results_visibility="private",
        )
    )
    df = results.select("scenario.replicate_id", "answer.output", "scenario.condition").to_pandas()
    df.rename(columns={
        "scenario.replicate_id": "replicate_id",
        "answer.output": "output",
        "scenario.condition": "condition",
    }, inplace=True)
    df["assistant_model"] = condition.replace("scaffold_", "")
    df["worker_model"] = base_worker_model
    safe_name = condition.replace(":", "_").replace("/", "_")
    Path(f"travel-agent-outputs/{TASK_NAME}/{safe_name}").mkdir(parents=True, exist_ok=True)
    df.to_csv(f"travel-agent-outputs/{TASK_NAME}/{safe_name}/outputs.csv", index=False)
    print(f"💾 Saved {condition} responses ({len(df)} rows)")
    return df

all_results = []
plain_df = run_condition(BASE_WORKER_MODEL, "plain", TASK_PROMPT)
all_results.append(plain_df)

for model_id, model_label in ASSISTANT_MODELS.items():
    scaffold_text = generate_scaffold(TASK_PROMPT, model_id)
    condition = f"scaffold_{model_label.replace(' ', '_')}"
    scaffold_df = run_condition(BASE_WORKER_MODEL, condition, TASK_PROMPT, scaffold_text=scaffold_text)
    all_results.append(scaffold_df)

final_df = pd.concat(all_results, ignore_index=True)
Path("travel-agent-results").mkdir(exist_ok=True)
output_path = "travel-agent-results/travel_agent_augmentation_results.csv"
final_df.to_csv(output_path, index=False)
print(f"\n✅ Exported all results → {output_path} ({len(final_df)} rows)")
print(final_df.head())

In [ ]:
# ======================================================
# Phase 3b Evaluation: Pairwise Grading + Average Rank
# ======================================================

from edsl import Agent, Model, Scenario, ScenarioList, Survey, QuestionMultipleChoice
import pandas as pd
import numpy as np
import itertools
from pathlib import Path

# ------------------------------------------------------
# Config
# ------------------------------------------------------
INPUT_FILE = "travel-agent-results/travel_agent_augmentation_results.csv"
OUTPUT_FILE = "travel-agent-results/travel_agent_pairwise_ranked.csv"
LEADERBOARD_FILE = "travel-agent-results/travel_agent_pairwise_leaderboard.csv"
RUN_LEVEL_FILE = "travel-agent-results/travel_agent_pairwise_run_level.csv"

EVALUATOR_MODEL = "gpt-5"

# repeated judgments per pair within one run
N_EVALS = 3

# repeat full tournament to compute SD across runs
N_RUNS = 5

# ------------------------------------------------------
# Task Context
# ------------------------------------------------------
TASK_TEXT = """
A customer tells you they want to take a 5-day trip to Tokyo next month with a $2,500 budget.

In one comprehensive response, do ALL of the following:

(1) Ask any missing clarifying questions needed to finalize plans.
(2) Compute estimated costs for flights, hotels, local transportation, and activities.
(3) Provide 2–3 hotel options with nightly rates.
(4) Suggest a transportation plan (airline choices, rail passes, transit tips).
(5) Provide brochures-style info: local customs, key points of interest, cultural etiquette, necessary regulations, passport/visa notes.
(6) Create a detailed 5-day itinerary with morning/afternoon/evening activities.
(7) Offer at least one promotional package or bundled tour option.
(8) Present a clear cost breakdown and confirm whether the trip fits the budget.

Respond exactly as a professional travel agent would.
"""

# ------------------------------------------------------
# Pairwise Evaluation Prompt
# ------------------------------------------------------
EVAL_PROMPT = """
You are an expert evaluator simulating a thoughtful human reviewing practical work.

Choose the better response overall.

Return only:
- option_1
or
- option_2
"""

# ------------------------------------------------------
# Scenario Builder
# ------------------------------------------------------
def build_pairwise_scenarios(df, n_evals, run_id):

    scenarios = []
    item_pairs = list(itertools.combinations(df.index.tolist(), 2))

    for i, j in item_pairs:

        row_i = df.loc[i]
        row_j = df.loc[j]

        for replicate_id in range(n_evals):

            scenarios.append(
                Scenario(
                    {
                        "run_id": run_id,
                        "left_idx": i,
                        "right_idx": j,
                        "replicate_id": replicate_id,
                        "task_text": TASK_TEXT,
                        "option_1": str(row_i["output"]),
                        "option_2": str(row_j["output"]),
                        "left_worker_model": row_i.get("worker_model", "unknown"),
                        "right_worker_model": row_j.get("worker_model", "unknown"),
                        "left_assistant_model": row_i.get("assistant_model", "none"),
                        "right_assistant_model": row_j.get("assistant_model", "none"),
                        "left_condition": row_i.get("condition", "unknown"),
                        "right_condition": row_j.get("condition", "unknown"),
                    }
                )
            )

    return ScenarioList(scenarios)

# ------------------------------------------------------
# Pairwise Evaluation (single run)
# ------------------------------------------------------
def evaluate_pairwise_once(df, run_id):

    df = df.copy().reset_index(drop=True)

    print(f"\n🚀 Run {run_id+1}/{N_RUNS}")
    print(f"Loaded {len(df)} candidate outputs")

    scenarios = build_pairwise_scenarios(df, N_EVALS, run_id)

    q = QuestionMultipleChoice(
        question_name="winner",
        question_text="""
Original request:
{{ task_text }}

--------------------------------------------------
OPTION 1
{{ option_1 }}

--------------------------------------------------
OPTION 2
{{ option_2 }}

Which option is better overall?
""",
        question_options=["option_1", "option_2"],
        include_comment=False,
    )

    survey = Survey([q])
    agent = Agent(instruction=EVAL_PROMPT)
    evaluator = Model(EVALUATOR_MODEL)

    results = (
        survey
        .by(scenarios)
        .by(agent)
        .by(evaluator)
        .run(progress_bar=False, verbose=False)
    )

    pairwise_df = results.select(
        "run_id",
        "left_idx",
        "right_idx",
        "replicate_id",
        "winner",
    ).to_pandas()

    pairwise_df.columns = [c.split(".")[-1] for c in pairwise_df.columns]

    return df, pairwise_df

# ------------------------------------------------------
# Ranking Aggregation (single run)
# ------------------------------------------------------
def aggregate_rankings_once(df, pairwise_df, run_id):

    records = []

    for replicate_id in range(N_EVALS):

        sub = pairwise_df[pairwise_df["replicate_id"] == replicate_id]

        wins = {i:0 for i in df.index}
        games = {i:0 for i in df.index}

        for _, row in sub.iterrows():

            left = int(row["left_idx"])
            right = int(row["right_idx"])

            games[left]+=1
            games[right]+=1

            if row["winner"]=="option_1":
                wins[left]+=1
            else:
                wins[right]+=1

        rep_df = pd.DataFrame({
            "run_id":run_id,
            "item_id":list(df.index),
            "wins":[wins[i] for i in df.index],
            "games":[games[i] for i in df.index]
        })

        rep_df["win_rate"]=rep_df["wins"]/rep_df["games"]
        rep_df["rank"]=rep_df["wins"].rank(ascending=False)

        records.append(rep_df)

    long_rank_df=pd.concat(records)

    run_summary=(
        long_rank_df
        .groupby("item_id")
        .agg(
            run_avg_rank=("rank","mean"),
            run_avg_win_rate=("win_rate","mean"),
            run_total_wins=("wins","sum"),
            run_total_games=("games","sum")
        )
        .reset_index()
    )

    scored=df.copy()
    scored["item_id"]=scored.index
    scored=scored.merge(run_summary,on="item_id")
    scored["run_id"]=run_id

    return scored,long_rank_df

# ------------------------------------------------------
# Run full experiment
# ------------------------------------------------------
def run_full_experiment(df):

    all_runs=[]
    all_pairwise=[]
    all_long=[]

    for run_id in range(N_RUNS):

        scored_base,pairwise_df=evaluate_pairwise_once(df,run_id)

        scored,long_rank_df=aggregate_rankings_once(
            scored_base,
            pairwise_df,
            run_id
        )

        all_runs.append(scored)
        all_pairwise.append(pairwise_df)
        all_long.append(long_rank_df)

    return (
        pd.concat(all_runs),
        pd.concat(all_long),
        pd.concat(all_pairwise)
    )

# ------------------------------------------------------
# Cross-run summary
# ------------------------------------------------------
def summarize_across_runs(df,all_scored_runs_df):

    base=df.copy().reset_index(drop=True)
    base["item_id"]=base.index

    cross_run_summary=(
        all_scored_runs_df
        .groupby("item_id")
        .agg(
            avg_rank=("run_avg_rank","mean"),
            sd_rank=("run_avg_rank","std"),
            avg_win_rate=("run_avg_win_rate","mean"),
            sd_win_rate=("run_avg_win_rate","std"),
            avg_total_wins=("run_total_wins","mean"),
            avg_total_games=("run_total_games","mean"),
            n_runs=("run_id","nunique")
        )
        .reset_index()
    )

    scored=base.merge(cross_run_summary,on="item_id")

    scored=scored.sort_values(
        ["avg_rank","avg_win_rate"],
        ascending=[True,False]
    )

    return scored

# ------------------------------------------------------
# Main
# ------------------------------------------------------
def main():

    df=pd.read_csv(INPUT_FILE)

    Path("travel-agent-results").mkdir(exist_ok=True)

    all_scored_runs_df,all_long_rank_df,all_pairwise_df=run_full_experiment(df)

    scored=summarize_across_runs(df,all_scored_runs_df)

    scored.to_csv(OUTPUT_FILE,index=False)
    print("Saved ranked results")

    all_scored_runs_df.to_csv(RUN_LEVEL_FILE,index=False)

    leaderboard = (
    all_scored_runs_df.groupby(["worker_model", "assistant_model", "condition"])
    .agg(
        avg_rank=("run_avg_rank", "mean"),
        sd_rank=("run_avg_rank", "std"),
        avg_win_rate=("run_avg_win_rate", "mean"),
        sd_win_rate=("run_avg_win_rate", "std"),
        avg_total_wins=("run_total_wins", "mean"),
        avg_total_games=("run_total_games", "mean"),
        n_runs=("run_id", "nunique"),
    )
    .reset_index()
    .sort_values(by=["avg_rank", "avg_win_rate"], ascending=[True, False])
)

    leaderboard.to_csv(LEADERBOARD_FILE, index=False)
    print(f"🏁 Leaderboard saved → {LEADERBOARD_FILE}")

    print("\nLeaderboard:")
    print(
        leaderboard[
            [
                "worker_model",
                "assistant_model",
                "condition",
                "avg_rank",
                "avg_win_rate",
                "sd_rank",
                "sd_win_rate",
                "n_runs",
            ]
        ].head(10)
)

if __name__=="__main__":
    main()

In [ ]:
# ============================================================
# 📊 Phase 3b Visualization — Scaffold Comparison (Win Rate)
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- Load run-level results ---
df = pd.read_csv("travel-agent-results/travel_agent_pairwise_run_level.csv")

# --- Normalize assistant names to match ASSISTANT_MODELS labels ---
assistant_name_map = {
    "gpt-4.1": "GPT-4.1",
    "deepseek-ai/DeepSeek-V3.1": "DeepSeek-V3.1",
    "o4-mini-2025-04-16": "GPT-O4-Mini",
    "o3-mini-2025-01-31": "GPT-O3-Mini",
    "openai/gpt-oss-120b": "GPT-OSS-120B",
    "anthropic.claude-3-5-sonnet-20240620-v1:0": "Claude-3.5-Sonnet",
    "gpt-5.4-2026-03-05": "GPT-5.4",
    "gpt-5.4": "GPT-5.4",
    "claude-opus-4-6": "Claude-Opus-4.6",
    "gemini-3.1-pro-preview": "Gemini-3.1-Pro",
    "plain": "No Scaffold",
}

df["assistant_model"] = df["assistant_model"].replace(assistant_name_map)

# --- Clean up condition for legend/grouping ---
df["condition"] = np.where(
    df["condition"] == "plain",
    "Plain",
    "Scaffolded"
)

# --- Optional: enforce plotting order to match ASSISTANT_MODELS ---
model_order = [
    "No Scaffold",
    "GPT-4.1",
    "DeepSeek-V3.1",
    "GPT-O4-Mini",
    "GPT-O3-Mini",
    "GPT-OSS-120B",
    "Claude-3.5-Sonnet",
    "GPT-5.4",
    "Claude-Opus-4.6",
    "Gemini-3.1-Pro",
]

df["assistant_model"] = pd.Categorical(
    df["assistant_model"],
    categories=model_order,
    ordered=True
)

# --- Convert run-level win rate to percent ---
df["run_avg_win_rate_pct"] = df["run_avg_win_rate"] * 100

# --- Sort for display ---
df = df.sort_values(by="assistant_model")

# --- Seaborn style ---
sns.set(style="whitegrid", context="talk", font_scale=1.1)
plt.figure(figsize=(14, 7))

# --- Barplot with automatic SD across runs ---
ax = sns.barplot(
    data=df,
    x="assistant_model",
    y="run_avg_win_rate_pct",
    hue="condition",
    errorbar="sd",
    palette={"Plain": "#4C72B0", "Scaffolded": "#55A868"},
)

# --- Add value labels from summary stats ---
summary_df = (
    df.groupby(["assistant_model", "condition"], observed=True)["run_avg_win_rate_pct"]
    .agg(["mean", "std"])
    .reset_index()
    .sort_values(["assistant_model", "condition"])
)

for bar, (_, row) in zip(ax.patches, summary_df.iterrows()):
    x = bar.get_x() + bar.get_width() / 2
    y = bar.get_height()
    sd = 0.0 if pd.isna(row["std"]) else row["std"]

    plt.text(
        x,
        y + 1.0,
        f"{row['mean']:.1f}%±{sd:.1f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

# --- Labels, title, legend ---
plt.title(
    "Augmentation Win Rate (Travel Agent)\n"
    "(Mean Pairwise Win Rate ± 1 SD Across Runs)",
    fontsize=17,
    weight="bold"
)
plt.ylabel("Average Pairwise Win Rate (%)", fontsize=13)
plt.xlabel("Assistant Scaffold Model", fontsize=12)
plt.xticks(rotation=30, ha="right", fontsize=11)
plt.ylim(0, 100)
plt.axhline(50, linestyle="--", color="gray", alpha=0.6)
plt.legend(title="Condition", title_fontsize=12, fontsize=11)
plt.tight_layout()

# --- Save & show ---
plt.savefig("travel_agent_scaffold_winrate.png", dpi=300, bbox_inches="tight")
plt.show()


# Automation

In [ ]:
# =========================================================
# Direct Model Comparison (Each Model Does the Task Itself)
# =========================================================
from edsl import Agent, Model, Scenario, ScenarioList, Survey, QuestionFreeText
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------
# 1. Experiment Configuration
# ---------------------------------------------------------
N_REPLICATES = 1

TASK_MODELS = {
    "gpt-3.5-turbo": "GPT-3.5-Turbo",
    "gpt-4.1": "GPT-4.1",
    "deepseek-ai/DeepSeek-V3.1": "DeepSeek-V3.1",
    "o4-mini-2025-04-16": "GPT-O4-Mini",
    "o3-mini-2025-01-31": "GPT-O3-Mini",
    "openai/gpt-oss-120b": "GPT-OSS-120B",
    "anthropic.claude-3-5-sonnet-20240620-v1:0": "Claude-3.5-Sonnet",
    "gpt-5.4": "gpt-5.4",
    "claude-opus-4-6": "Claude-Opus-4.6",
    "gemini-3.1-pro-preview": "Gemini-3.1-Pro",
}

# ---------------------------------------------------------
# 2. Task
# ---------------------------------------------------------
TASK_NAME = "travel_agent"
TASK_PROMPT = """
A customer tells you they want to take a 5-day trip to Tokyo next month with a $2,500 budget.

In one comprehensive response, do ALL of the following:

(1) Ask any missing clarifying questions needed to finalize plans.

(2) Compute estimated costs for flights, hotels, local transportation, and activities.

(3) Provide 2–3 hotel options with nightly rates.

(4) Suggest a transportation plan (airline choices, rail passes, transit tips).

(5) Provide brochures-style info: local customs, key points of interest, cultural etiquette, necessary regulations, passport/visa notes.

(6) Create a detailed 5-day itinerary with morning/afternoon/evening activities.

(7) Offer at least one promotional package or bundled tour option.

(8) Present a clear cost breakdown and confirm whether the trip fits the budget.

Respond exactly as a professional travel agent would. Be thorough, professional, and customer-focused. Provide realistic and practical recommendations that fit within the customer's budget.
"""

# ---------------------------------------------------------
# 3. Run one model directly on the task
# ---------------------------------------------------------
def run_model_condition(model_id, model_label, task_prompt):
    scenarios = ScenarioList([
        Scenario({
            "task_prompt": task_prompt,
            "condition": model_label,
            "replicate_id": i
        })
        for i in range(N_REPLICATES)
    ])

    q = QuestionFreeText("output", "{{ scenario.task_prompt }}")
    survey = Survey([q])

    worker = Agent(
        instruction=(
            "Complete the task clearly and professionally. "
            "Return only the final deliverable."
        )
    )

    print(f"🚀 Running {model_id} [{model_label}]...")

    results = (
        survey.by(scenarios)
        .by(worker)
        .by(Model(model_id, max_tokens=4096))
        .run(
            n=1,
            progress_bar=False,
            verbose=False,
            print_exceptions=True,
            stop_on_exception=False,
            check_api_keys=False,
        )
    )

    df = results.select(
        "scenario.replicate_id",
        "answer.output",
        "scenario.condition"
    ).to_pandas()

    df.rename(columns={
        "scenario.replicate_id": "replicate_id",
        "answer.output": "output",
        "scenario.condition": "condition",
    }, inplace=True)

    df["model_id"] = model_id
    df["model_label"] = model_label

    safe_name = model_label.replace(" ", "_").replace(":", "_").replace("/", "_")
    Path(f"travel-agent-outputs/automation/{TASK_NAME}/{safe_name}").mkdir(parents=True, exist_ok=True)
    df.to_csv(f"travel-agent-outputs/automation/{TASK_NAME}/{safe_name}/outputs.csv", index=False)

    print(f"💾 Saved {model_label} responses ({len(df)} rows)")
    return df
# ---------------------------------------------------------
# 4. Main
# ---------------------------------------------------------
all_results = []

for model_id, model_label in TASK_MODELS.items():
    model_df = run_model_condition(model_id, model_label, TASK_PROMPT)
    all_results.append(model_df)

final_df = pd.concat(all_results, ignore_index=True)

Path("travel-agent-results").mkdir(exist_ok=True)
output_path = "travel-agent-results/travel_agent_automation_models.csv"
final_df.to_csv(output_path, index=False)

print(f"\n✅ Exported all results → {output_path} ({len(final_df)} rows)")
print(final_df[["replicate_id", "condition", "model_label", "output"]].head())

In [ ]:
from edsl import Agent, Model, Scenario, ScenarioList, Survey, QuestionMultipleChoice
import pandas as pd
import numpy as np
import itertools
from pathlib import Path

# ======================================================
# Phase 3b Evaluation: Automation Models Pairwise Grading
# ======================================================

# ------------------------------------------------------
# Config
# ------------------------------------------------------
INPUT_FILE = "travel-agent-results/travel_agent_automation_models.csv"
OUTPUT_FILE = "travel-agent-results/travel_agent_automation_pairwise_ranked.csv"
LEADERBOARD_FILE = "travel-agent-results/travel_agent_automation_pairwise_leaderboard.csv"
RUN_LEVEL_FILE = "travel-agent-results/travel_agent_automation_pairwise_run_level.csv"

EVALUATOR_MODEL_NAME = "gpt-5"

# Inner repeat: judgments per pair inside one tournament
N_EVALS = 3

# Outer repeat: rerun whole tournament to estimate SD across runs
N_RUNS = 5

# ------------------------------------------------------
# Task Context
# ------------------------------------------------------
TASK_TEXT = """
A customer tells you they want to take a 5-day trip to Tokyo next month with a $2,500 budget.

In one comprehensive response, do ALL of the following:

(1) Ask any missing clarifying questions needed to finalize plans.

(2) Compute estimated costs for flights, hotels, local transportation, and activities.

(3) Provide 2–3 hotel options with nightly rates.

(4) Suggest a transportation plan (airline choices, rail passes, transit tips).

(5) Provide brochures-style info: local customs, key points of interest, cultural etiquette, necessary regulations, passport/visa notes.

(6) Create a detailed 5-day itinerary with morning/afternoon/evening activities.

(7) Offer at least one promotional package or bundled tour option.

(8) Present a clear cost breakdown and confirm whether the trip fits the budget.

Respond exactly as a professional travel agent would. Be thorough, professional, and customer-focused. Provide realistic and practical recommendations that fit within the customer's budget.
"""

# ------------------------------------------------------
# Pairwise Evaluation Prompt
# ------------------------------------------------------
EVAL_PROMPT = """
You are an expert evaluator simulating a thoughtful human reviewing practical work.

You will receive:
1. The original task request (the Tokyo travel agent prompt)
2. Two candidate travel itineraries/responses

Choose the one that is better overall for actual use.

Consider silently:
- Does it explicitly address all 8 required components (clarifying questions, cost estimates, hotel options, transport plan, brochure info, 5-day itinerary, promo option, and a final budget breakdown)?
- Are the cost estimates and budget feasibility realistic for a 5-day trip to Tokyo on a $2,500 budget?
- Is the 5-day itinerary logically structured, practical, and broken down into morning/afternoon/evening activities?
- Does the tone feel genuinely professional, thorough, and customer-focused, like a real travel agent consulting a client?
- Which response is better organized, easier for a customer to digest, and ultimately more helpful for planning their trip?

Return only:
- option_1
or
- option_2

No reasoning. No extra text.
"""

# ------------------------------------------------------
# Helpers
# ------------------------------------------------------
def build_pairwise_scenarios(df, n_evals, run_id):
    scenarios = []
    item_pairs = list(itertools.combinations(df.index.tolist(), 2))

    for i, j in item_pairs:
        row_i = df.loc[i]
        row_j = df.loc[j]

        if pd.isna(row_i["output"]) or pd.isna(row_j["output"]):
            continue

        for replicate_id in range(n_evals):
            scenarios.append(
                Scenario(
                    {
                        "run_id": int(run_id),
                        "left_idx": int(i),
                        "right_idx": int(j),
                        "replicate_id": int(replicate_id),
                        "task_text": str(TASK_TEXT),
                        "option_1": str(row_i["output"]),
                        "option_2": str(row_j["output"]),
                        "left_model_id": str(row_i.get("model_id", "unknown")),
                        "right_model_id": str(row_j.get("model_id", "unknown")),
                        "left_model_label": str(row_i.get("model_label", "unknown")),
                        "right_model_label": str(row_j.get("model_label", "unknown")),
                        "left_condition": str(row_i.get("condition", "unknown")),
                        "right_condition": str(row_j.get("condition", "unknown")),
                        "left_replicate_id": (
                            int(row_i["replicate_id"])
                            if pd.notna(row_i.get("replicate_id"))
                            else None
                        ),
                        "right_replicate_id": (
                            int(row_j["replicate_id"])
                            if pd.notna(row_j.get("replicate_id"))
                            else None
                        ),
                    }
                )
            )

    return ScenarioList(scenarios)


def evaluate_pairwise_once(df, run_id, n_evals=N_EVALS):
    df = df.copy().reset_index(drop=True)
    df = df[df["output"].notna()].reset_index(drop=True)

    print(f"\nRun {run_id + 1}/{N_RUNS}")
    print(f"Loaded {len(df)} candidate outputs")
    print(f"Building all pairwise comparisons x {n_evals} replicates")

    scenarios = build_pairwise_scenarios(df, n_evals=n_evals, run_id=run_id)

    q = QuestionMultipleChoice(
        question_name="winner",
        question_text="""
Original request:
{{ task_text }}

--------------------------------------------------
OPTION 1
{{ option_1 }}

--------------------------------------------------
OPTION 2
{{ option_2 }}

Which option is better overall?
""",
        question_options=["option_1", "option_2"],
        include_comment=False,
    )

    survey = Survey([q])
    agent = Agent(instruction=EVAL_PROMPT)
    evaluator = Model(EVALUATOR_MODEL_NAME)

    results = (
        survey.by(scenarios)
        .by(agent)
        .by(evaluator)
        .run(
            n=1,
            progress_bar=False,
            verbose=False,
            stop_on_exception=False,
            check_api_keys=False,
            print_exceptions=True,
        )
    )

    pairwise_df = results.select(
        "run_id",
        "left_idx",
        "right_idx",
        "replicate_id",
        "winner",
    ).to_pandas()

    pairwise_df.columns = [c.split(".")[-1] for c in pairwise_df.columns]

    return df, pairwise_df


def aggregate_rankings_once(df, pairwise_df, run_id, n_evals=N_EVALS):
    records = []

    for replicate_id in range(n_evals):
        sub = pairwise_df[pairwise_df["replicate_id"] == replicate_id].copy()

        wins = {i: 0 for i in df.index}
        games = {i: 0 for i in df.index}

        for _, row in sub.iterrows():
            left = int(row["left_idx"])
            right = int(row["right_idx"])
            winner = str(row["winner"]).strip()

            games[left] += 1
            games[right] += 1

            if winner == "option_1":
                wins[left] += 1
            elif winner == "option_2":
                wins[right] += 1

        rep_df = pd.DataFrame(
            {
                "run_id": run_id,
                "item_id": list(df.index),
                "replicate_id": replicate_id,
                "wins": [wins[i] for i in df.index],
                "games": [games[i] for i in df.index],
            }
        )
        rep_df["win_rate"] = rep_df["wins"] / rep_df["games"].replace(0, np.nan)
        rep_df["rank"] = rep_df["wins"].rank(ascending=False, method="average")

        records.append(rep_df)

    long_rank_df = pd.concat(records, ignore_index=True)

    run_summary = (
        long_rank_df.groupby("item_id")
        .agg(
            run_avg_rank=("rank", "mean"),
            run_std_rank=("rank", "std"),
            run_avg_win_rate=("win_rate", "mean"),
            run_std_win_rate=("win_rate", "std"),
            run_total_wins=("wins", "sum"),
            run_total_games=("games", "sum"),
        )
        .reset_index()
    )

    scored = df.copy()
    scored["item_id"] = scored.index
    scored = scored.merge(run_summary, on="item_id", how="left")
    scored["run_id"] = run_id

    scored = scored.sort_values(
        ["run_avg_rank", "run_avg_win_rate"],
        ascending=[True, False]
    )

    return scored, long_rank_df


def run_full_experiment(df, n_runs=N_RUNS, n_evals=N_EVALS):
    all_scored_runs = []
    all_long_rank_dfs = []
    all_pairwise_dfs = []

    for run_id in range(n_runs):
        scored_base, pairwise_df = evaluate_pairwise_once(
            df,
            run_id=run_id,
            n_evals=n_evals,
        )
        scored_run, long_rank_df = aggregate_rankings_once(
            scored_base,
            pairwise_df,
            run_id=run_id,
            n_evals=n_evals,
        )

        all_pairwise_dfs.append(pairwise_df)
        all_long_rank_dfs.append(long_rank_df)
        all_scored_runs.append(scored_run)

    all_pairwise_df = pd.concat(all_pairwise_dfs, ignore_index=True)
    all_long_rank_df = pd.concat(all_long_rank_dfs, ignore_index=True)
    all_scored_runs_df = pd.concat(all_scored_runs, ignore_index=True)

    return all_scored_runs_df, all_long_rank_df, all_pairwise_df


def summarize_across_runs(df, all_scored_runs_df):
    base = df.copy().reset_index(drop=True)
    base = base[base["output"].notna()].reset_index(drop=True)
    base["item_id"] = base.index

    cross_run_summary = (
        all_scored_runs_df.groupby("item_id")
        .agg(
            avg_rank=("run_avg_rank", "mean"),
            sd_rank=("run_avg_rank", "std"),
            avg_win_rate=("run_avg_win_rate", "mean"),
            sd_win_rate=("run_avg_win_rate", "std"),
            avg_total_wins=("run_total_wins", "mean"),
            sd_total_wins=("run_total_wins", "std"),
            avg_total_games=("run_total_games", "mean"),
            n_runs=("run_id", "nunique"),
        )
        .reset_index()
    )

    scored = base.merge(cross_run_summary, on="item_id", how="left")
    scored = scored.sort_values(["avg_rank", "avg_win_rate"], ascending=[True, False])

    return scored


# ======================================================
# Main
# ======================================================
def main():
    df = pd.read_csv(INPUT_FILE)
    Path("travel-agent-results").mkdir(exist_ok=True)

    all_scored_runs_df, all_long_rank_df, all_pairwise_df = run_full_experiment(
        df,
        n_runs=N_RUNS,
        n_evals=N_EVALS,
    )

    scored = summarize_across_runs(df, all_scored_runs_df)
    scored.to_csv(OUTPUT_FILE, index=False)
    print(f"Saved pairwise-ranked results -> {OUTPUT_FILE}")

    all_scored_runs_df.to_csv(RUN_LEVEL_FILE, index=False)
    print(f"Saved per-run results -> {RUN_LEVEL_FILE}")

    leaderboard = (
        all_scored_runs_df.groupby(["model_id", "model_label", "condition"])
        .agg(
            avg_rank=("run_avg_rank", "mean"),
            sd_rank=("run_avg_rank", "std"),
            avg_win_rate=("run_avg_win_rate", "mean"),
            sd_win_rate=("run_avg_win_rate", "std"),
            avg_total_wins=("run_total_wins", "mean"),
            avg_total_games=("run_total_games", "mean"),
            n_runs=("run_id", "nunique"),
        )
        .reset_index()
        .sort_values(by=["avg_rank", "avg_win_rate"], ascending=[True, False])
    )

    leaderboard.to_csv(LEADERBOARD_FILE, index=False)
    print(f"Saved leaderboard -> {LEADERBOARD_FILE}")

    print("\nTop ranked outputs:")
    print(
        scored[
            [
                "model_label",
                "condition",
                "replicate_id",
                "avg_rank",
                "sd_rank",
                "avg_win_rate",
                "sd_win_rate",
                "avg_total_wins",
                "n_runs",
            ]
        ].head(10)
    )

    print("\nLeaderboard:")
    print(
        leaderboard[
            [
                "model_label",
                "condition",
                "avg_rank",
                "avg_win_rate",
                "sd_rank",
                "sd_win_rate",
                "n_runs",
            ]
        ].head(10)
    )


if __name__ == "__main__":
    main()

In [ ]:
# ============================================================
# 📊 Phase 3b Visualization — Automation Model Comparison
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load run-level results ---
df = pd.read_csv("travel-agent-results/travel_agent_automation_pairwise_run_level.csv")

# --- Normalize model names to match TASK_MODELS labels ---
model_name_map = {
    "gpt-3.5-turbo": "GPT-3.5-Turbo",
    "gpt-4.1": "GPT-4.1",
    "deepseek-ai/DeepSeek-V3.1": "DeepSeek-V3.1",
    "o4-mini-2025-04-16": "GPT-O4-Mini",
    "o3-mini-2025-01-31": "GPT-O3-Mini",
    "openai/gpt-oss-120b": "GPT-OSS-120B",
    "anthropic.claude-3-5-sonnet-20240620-v1:0": "Claude-3.5-Sonnet",
    "gpt-5.4-2026-03-05": "GPT-5.4",
    "gpt-5.4": "GPT-5.4",
    "claude-opus-4-6": "Claude-Opus-4.6",
    "gemini-3.1-pro-preview": "Gemini-3.1-Pro",
}

if "model_label" in df.columns:
    df["plot_model"] = df["model_label"].replace(model_name_map)
elif "model_id" in df.columns:
    df["plot_model"] = df["model_id"].replace(model_name_map)
else:
    raise ValueError("Expected either 'model_label' or 'model_id' column in run-level file.")

# --- Optional: enforce plotting order to match TASK_MODELS ---
model_order = [
    "GPT-3.5-Turbo",
    "GPT-4.1",
    "DeepSeek-V3.1",
    "GPT-O4-Mini",
    "GPT-O3-Mini",
    "GPT-OSS-120B",
    "Claude-3.5-Sonnet",
    "GPT-5.4",
    "Claude-Opus-4.6",
    "Gemini-3.1-Pro",
]

df["plot_model"] = pd.Categorical(
    df["plot_model"],
    categories=model_order,
    ordered=True
)

# --- Convert run-level win rate to percent ---
df["run_avg_win_rate_pct"] = df["run_avg_win_rate"] * 100

# --- Sort for display ---
df = df.sort_values(by="plot_model")

# --- Seaborn style ---
sns.set(style="whitegrid", context="talk", font_scale=1.1)
plt.figure(figsize=(14, 7))

# --- Barplot with automatic SD across runs ---
ax = sns.barplot(
    data=df,
    x="plot_model",
    y="run_avg_win_rate_pct",
    color="#8172B3",
    errorbar="sd",
)

# --- Add value labels from summary stats ---
summary_df = (
    df.groupby("plot_model", observed=True)["run_avg_win_rate_pct"]
    .agg(["mean", "std"])
    .reset_index()
    .sort_values("plot_model")
)

for bar, (_, row) in zip(ax.patches, summary_df.iterrows()):
    x = bar.get_x() + bar.get_width() / 2
    y = bar.get_height()
    sd = 0.0 if pd.isna(row["std"]) else row["std"]

    plt.text(
        x,
        y + 1.0,
        f"{row['mean']:.1f}%±{sd:.1f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

# --- Labels and title ---
plt.title(
    "Automation Model Performance on Travel Agent Task\n"
    "(Mean Pairwise Win Rate ± 1 SD Across Runs)",
    fontsize=17,
    weight="bold"
)
plt.ylabel("Average Pairwise Win Rate (%)", fontsize=13)
plt.xlabel("Automation Model", fontsize=12)
plt.xticks(rotation=30, ha="right", fontsize=11)
plt.ylim(0, 100)

# --- Optional reference line at chance level ---
plt.axhline(50, linestyle="--", color="gray", alpha=0.6)

plt.tight_layout()

# --- Save & show ---
plt.savefig("travel_agent_automation_winrate.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import shutil
from google.colab import files
import os

output_folder = "travel-agent-outputs"
results_folder = "travel-agent-results"
zip_name = "travel-planning"

# Create the zip archive
if os.path.exists(output_folder):
    shutil.make_archive(zip_name + "_outputs", 'zip', output_folder)
    print(f"Created {zip_name}_outputs.zip")
    files.download(f"{zip_name}_outputs.zip")
else:
    print(f"Folder '{output_folder}' does not exist.")

if os.path.exists(results_folder):
    shutil.make_archive(zip_name + "_results", 'zip', results_folder)
    print(f"Created {zip_name}_results.zip")
    files.download(f"{zip_name}_results.zip")
else:
    print(f"Folder '{results_folder}' does not exist.")

print("Download process initiated. Please check your browser for the downloads.")